# Solving Non-Singular Systems of Equations

*Course notes for **Math for Machine Learning**, C1 · W2 · L1 — "Solving Non-Singular Systems of Equations" (DeepLearning.AI).*

Week 1 told us *when* a system has a unique solution (it must be **non-singular**). Week 2 starts on *how* to find it. The strategy is **elimination**: manipulate the equations until each variable stands alone.

$$ \text{System} \;\xrightarrow{\;\text{manipulate equations}\;}\; \text{Solved system } (a = \dots,\; b = \dots). $$

Two manipulations keep the solution unchanged:

1. **Multiply (or divide) an equation by a non-zero constant.**
2. **Add (or subtract) one equation to/from another.**

In [1]:
import numpy as np
np.set_printoptions(precision=4, suppress=True)

## 1. The two legal moves

**Multiplying by a constant** — e.g. multiply $a + b = 10$ by $7$:
$$ a + b = 10 \;\xrightarrow{\times 7}\; 7a + 7b = 70. $$

**Adding two equations** — add them term by term to combine information (and, chosen well, cancel a variable).

Both produce an equivalent system: any solution of the original still satisfies the result.

## 2. Worked example

$$ \begin{cases} 5a + b = 17 \\ 4a - 3b = 6 \end{cases} $$

**Step 1 — divide each equation by its coefficient of $a$** (so $a$ has coefficient 1 everywhere):

$$ a + 0.2\,b = 3.4 \qquad a - 0.75\,b = 1.5 $$

**Step 2 — eliminate $a$** by subtracting equation 1 from equation 2:

$$ (a - 0.75b) - (a + 0.2b) = 1.5 - 3.4 \;\Rightarrow\; -0.95\,b = -1.9 \;\Rightarrow\; b = 2. $$

**Step 3 — back-substitute** $b=2$ into $a + 0.2b = 3.4$:

$$ a + 0.2(2) = 3.4 \;\Rightarrow\; a = 3. $$

In [2]:
# Reproduce the elimination by hand, step by step.
# Each equation is stored as [coeff_a, coeff_b, rhs].
eq1 = np.array([5.0, 1.0, 17.0])
eq2 = np.array([4.0, -3.0, 6.0])

# Step 1: divide each row by its coefficient of a.
r1 = eq1 / eq1[0]
r2 = eq2 / eq2[0]
print("Step 1  row1:", r1, "  row2:", r2)

# Step 2: subtract row1 from row2 to eliminate a, then solve for b.
r2 = r2 - r1
b = r2[2] / r2[1]
print("Step 2  eliminated row:", r2, " ->  b =", b)

# Step 3: back-substitute into row1 to get a.
a = r1[2] - r1[1] * b
print("Step 3  a =", a)

print("\nSolution: a =", a, ", b =", b)

Step 1  row1: [1.  0.2 3.4]   row2: [ 1.   -0.75  1.5 ]
Step 2  eliminated row: [ 0.   -0.95 -1.9 ]  ->  b = 2.0
Step 3  a = 3.0

Solution: a = 3.0 , b = 2.0


In [3]:
# Cross-check with numpy's solver.
A = np.array([[5.0, 1.0], [4.0, -3.0]])
d = np.array([17.0, 6.0])
print("numpy solution [a, b] =", np.linalg.solve(A, d))

numpy solution [a, b] = [3. 2.]


## 3. What if a coefficient of $a$ is zero?

$$ \begin{cases} 5a + b = 17 \\ \phantom{5a +\,} 3b = 6 \end{cases} $$

We can't "divide equation 2 by its coefficient of $a$" — that coefficient is **0**. But this is not a problem: $a$ is *already eliminated* from equation 2, which immediately gives $b = 2$. Back-substitute into equation 1: $5a + 2 = 17 \Rightarrow a = 3$.

The lesson: a zero in the pivot position just means we solve that equation directly (or swap rows). The system is still non-singular and solvable.

In [4]:
# Equation 2 has no 'a' -> read b off directly, then back-substitute.
b = 6 / 3
a = (17 - 1 * b) / 5
print(f"b = {b},  a = {a}")
print("numpy:", np.linalg.solve([[5.0, 1.0], [0.0, 3.0]], [17.0, 6.0]))

b = 2.0,  a = 3.0
numpy: [3. 2.]


## 4. Quiz

$$ \begin{cases} 2a + 5b = 46 \\ 8a + \phantom{5}b = 32 \end{cases} $$

Solve using the same elimination steps.

In [5]:
q1 = np.array([2.0, 5.0, 46.0])
q2 = np.array([8.0, 1.0, 32.0])

r1, r2 = q1 / q1[0], q2 / q2[0]   # normalise coefficient of a to 1
r2 = r2 - r1                       # eliminate a
b = r2[2] / r2[1]
a = r1[2] - r1[1] * b              # back-substitute
print(f"a = {a:.0f}, b = {b:.0f}")   # expected a = 3, b = 8

# Verify in the original equations.
print("check:", 2 * a + 5 * b, "= 46 ?", " ", 8 * a + b, "= 32 ?")

a = 3, b = 8
check: 46.0 = 46 ?   32.0 = 32 ?


## 5. Takeaways

- A non-singular system is solved by **elimination**: reduce it with legal moves until each variable is isolated.
- The two solution-preserving moves: **scale an equation** by a non-zero constant, and **add/subtract equations**.
- The recipe: normalise a pivot variable to coefficient 1 → subtract to **eliminate** it from other equations → **back-substitute** to recover the rest.
- A **zero pivot** is not a dead end — that equation is already simpler; solve it directly or swap rows.
- This hand procedure is exactly **Gaussian elimination**, the engine inside `numpy.linalg.solve`.

*Related:* the singular case (no unique solution) is handled differently — see [Singular vs Non-Singular Matrices](./C1_W1_L1_singular_vs_nonsingular_matrices.ipynb).